In [2]:
!pip install nsepython

In [12]:
import nsepython
import pandas

# df = nsepython.nse_get_advances_declines("pandas")

In [6]:
# print all headers from nse_get_advances_declines("pandas")
print(df.columns.tolist())

['symbol', 'identifier', 'series', 'open', 'dayHigh', 'dayLow', 'lastPrice', 'previousClose', 'change', 'pChange', 'totalTradedVolume', 'totalTradedValue', 'yearHigh', 'yearLow', 'nearWKH', 'nearWKL', 'perChange365d', 'perChange30d', 'date365dAgo', 'date30dAgo', 'chartTodayPath', 'chart30dPath', 'chart365dPath', 'meta']


In [8]:
# print data in meta column
print(df["meta"].tolist()[0])

{'symbol': 'RELIANCE', 'companyName': 'Reliance Industries Limited', 'industry': 'Refineries & Marketing', 'activeSeries': ['EQ', 'T0'], 'debtSeries': [], 'isFNOSec': True, 'isCASec': False, 'isSLBSec': True, 'isDebtSec': False, 'isSuspended': False, 'tempSuspendedSeries': [], 'isETFSec': False, 'isDelisted': False, 'isin': 'INE002A01018', 'slb_isin': 'INE002A01018', 'listingDate': '1995-11-29', 'isMunicipalBond': False, 'isHybridSymbol': False, 'segment': 'EQUITY', 'quotepreopenstatus': {'equityTime': '05-Mar-2026 16:00:00', 'preOpenTime': '05-Mar-2026 09:07:52', 'QuotePreOpenFlag': False}}


In [10]:
print(type(nsepython.nse_largedeals()))

<class 'pandas.core.frame.DataFrame'>


In [11]:
# save largedeals data to csv
nsepython.nse_largedeals().to_csv("largedeals.csv", index=False)


In [5]:
import time
from io import StringIO, BytesIO
import pandas as pd
import requests


def fetch_nse_historical_csv(symbol: str, from_date: str, to_date: str, csv: bool = True, session=None, headers=None, warmup: bool = True):
    """Fetch historical CSV from NSE and return a cleaned pandas DataFrame.
    Dates must be in DD-MM-YYYY format.
    """
    
    if headers is None:
        headers = {
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept": "text/csv,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://www.nseindia.com/",
            "Connection": "keep-alive",
        }

    url = f"https://www.nseindia.com/api/historicalOR/priceAndVolumeDataPerSecurity?symbol={symbol}&from={from_date}&to={to_date}&csv={'true' if csv else 'false'}"

    sess = session or requests.Session()
    sess.headers.update(headers)
    if warmup and session is None:
        try:
            sess.get("https://www.nseindia.com/", timeout=10)
        except Exception:
            pass
        time.sleep(0.8)

    resp = sess.get(url, timeout=20)
    resp.raise_for_status()

    try:
        df = pd.read_csv(BytesIO(resp.content), encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(StringIO(resp.text), encoding="utf-8-sig")

    # clean column names (remove BOM, surrounding quotes, extra spaces)
    df.columns = df.columns.str.lstrip('\ufeff').str.strip().str.replace('\"', '', regex=False)

    return df

# Example call:
# df = fetch_nse_historical_csv("SBIN", "01-01-2024", "14-06-2024")
# print(df.head())

In [6]:
# Use the helper function to fetch data and print results
df = fetch_nse_historical_csv("SBIN", "01-01-2024", "14-06-2024")
print(df.columns.tolist())
print(df.head())

['Symbol', 'Date', 'PREV. CLOSE', 'Open Price', 'High Price', 'Low Price', 'Close Price', 'Total Traded Quantity', 'Total Traded Value in Rs.', 'Number of Trades']
  Symbol         Date  PREV. CLOSE  Open Price  High Price  Low Price  \
0   SBIN  14-JUN-2024         0.94        0.50        0.50       0.50   
1   SBIN  14-JUN-2024         0.25        0.25        0.25       0.25   
2   SBIN  11-JUN-2024         0.45        0.98        0.98       0.98   
3   SBIN  10-JUN-2024         0.25        0.40        0.50       0.40   
4   SBIN  06-JUN-2024         0.25        0.25        0.25       0.25   

   Close Price Total Traded Quantity Total Traded Value in Rs.  \
0         0.50                 1,675                    838.00   
1         0.25                   755                    189.00   
2         0.98                   445                    436.00   
3         0.45                 4,691                  2,140.00   
4         0.25                 4,000                  1,000.00   

